# PPC v9 — SN-GAN Refiner for Gap-Perfect Point Clouds

Stage 2: Frozen v8 → SN-GAN discriminator + Point Refiner → Clean gaps


In [1]:
"""
PPC v9 — GAN Refiner for Gap-Perfect Point Clouds
==================================================
Stage 2: Takes pre-trained v8 output, adds WGAN-GP discriminator
to learn structural gap patterns that explicit losses cannot capture.

Pipeline:
  1. Load frozen v8 generator (PPCNetV8)
  2. PointRefiner: lightweight residual network (predicts per-point offsets)
  3. DGCNN Discriminator: classifies real GT vs refined predictions
  4. WGAN-GP training with reconstruction + adversarial loss
"""
import os, sys, json, time, math, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk

warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    fraction = min(50.0 / total_gb, 0.95)
    torch.cuda.set_per_process_memory_fraction(fraction, device=0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = Path('/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037')
SPLIT_FILE   = DATA_ROOT / 'dataset_split.json'
V8_CKPT_DIR  = Path('/apps/app/pandu/ppc_network_v8/checkpoints')
PROJECT_DIR  = Path('/apps/app/pandu/ppc_network_v9_gan')
CKPT_DIR     = PROJECT_DIR / 'checkpoints'
LOG_DIR      = PROJECT_DIR / 'logs'
RESULTS_DIR  = PROJECT_DIR / 'results'
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
IMG_SIZE        = 512
COARSE_VOL_SIZE = 32
AUX_VOL_SIZE    = 48
GAP_VOL_SIZE    = 96
N_POINTS        = 5120
ENC_CHANNELS    = 192
VOL_CHANNELS    = 128
DEC_CHANNELS    = 128
QUERY_DIM       = 256
N_REFINE_STAGES = 3
PRETRAINED      = True

# GAN training config
BATCH_SIZE      = 2
NUM_WORKERS     = 4
GAN_EPOCHS      = 150
LR_G            = 2e-5      # refiner learning rate
LR_D            = 1e-4      # discriminator learning rate
N_CRITIC        = 3         # D updates per G update
D_SUBSAMPLE     = 1024      # subsample points for discriminator (saves memory)
W_ADV           = 0.10      # adversarial loss weight
W_CHAMFER_R     = 1.00      # chamfer reconstruction
W_GAP_R         = 0.60      # gap penalty (stronger than v8)
W_AXIAL_R       = 0.50      # axial density matching
W_SW_R          = 0.30      # sliced wasserstein
USE_AMP         = True
GRAD_CLIP       = 1.0
MAX_EVAL        = 103

CKPT_PATH      = CKPT_DIR / 'latest_checkpoint.pth'
BEST_CKPT_PATH = CKPT_DIR / 'best_checkpoint.pth'
TRAINING_LOG   = LOG_DIR  / 'training_log.json'

print('='*60)
print('PPC v9 — GAN Refiner')
print('='*60)

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
PPC v9 — GAN Refiner


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UTILITIES (same as v8)
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    n = poly.GetNumberOfPoints()
    return np.array([poly.GetPoint(i) for i in range(n)], dtype=np.float32)

def save_vtk_points(points, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for p in points:
        vp.InsertNextPoint(float(p[0]), float(p[1]), float(p[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)):
        vc.InsertNextCell(1)
        vc.InsertCellPoint(i)
    poly = vtk.vtkPolyData()
    poly.SetPoints(vp)
    poly.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(output_path))
    w.SetInputData(poly)
    w.SetFileTypeToASCII()
    w.Write()

def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

def load_projection_matrix(path):
    with open(path, 'r') as f:
        vals = [float(v) for v in f.read().split()]
    return np.array(vals, dtype=np.float32).reshape(3, 4)

def load_split(split_path=SPLIT_FILE):
    with open(split_path, 'r') as f:
        d = json.load(f)
    return d['train'], d['val'], d['test']

def points_to_local(points, center, scale):
    return (points - center[None]) / (scale[None] + 1e-6)

def local_to_world(pts, center, scale):
    return pts * scale[:, None, :] + center[:, None, :]

def compute_scale(gt_full):
    extent = (gt_full.max(0) - gt_full.min(0)).astype(np.float32)
    scale = extent * 0.55 + np.array([20., 20., 30.], dtype=np.float32)
    return np.maximum(scale, np.array([50., 50., 80.], dtype=np.float32))

def make_aux_occupancy(pts_local, vol_size=AUX_VOL_SIZE, dilate=1):
    pts = np.clip((np.asarray(pts_local, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size,)*3, dtype=np.float32)
    for dx in range(-dilate, dilate+1):
        for dy in range(-dilate, dilate+1):
            for dz in range(-dilate, dilate+1):
                occ[np.clip(idx[:,0]+dx,0,vol_size-1),
                    np.clip(idx[:,1]+dy,0,vol_size-1),
                    np.clip(idx[:,2]+dz,0,vol_size-1)] = 1.0
    return occ

def make_gap_occupancy(pts_local, vol_size=GAP_VOL_SIZE):
    pts = np.clip((np.asarray(pts_local, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size,)*3, dtype=np.float32)
    occ[idx[:,0], idx[:,1], idx[:,2]] = 1.0
    return occ

def flip_projection_matrix_horizontal(P, img_size=IMG_SIZE):
    F_mat = np.array([[-1,0,img_size-1],[0,1,0],[0,0,1]], dtype=np.float32)
    return F_mat @ P

def append_training_log(log_path, rec):
    log = {'records': []}
    if log_path.exists():
        with open(log_path) as f: log = json.load(f)
    log['records'].append(rec)
    tmp = log_path.with_suffix('.json.tmp')
    with open(tmp, 'w') as f: json.dump(log, f, indent=2)
    tmp.replace(log_path)


class LumbarDatasetV9(Dataset):
    def __init__(self, patient_ids, data_root=DATA_ROOT, augment=False):
        self.patient_ids = patient_ids
        self.data_root = Path(data_root)
        self.augment = augment
        self.img_norm = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self): return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        pdir = self.data_root / pid
        drr_ap = load_drr_image(pdir/'AP_0'/'drr_AP_0.png')
        drr_lp = load_drr_image(pdir/'LP_90'/'drr_LP_90.png')
        P_ap = load_projection_matrix(pdir/'AP_0'/'P_AP_0.txt')
        P_lp = load_projection_matrix(pdir/'LP_90'/'P_LP_90.txt')
        gt_full = load_vtk_points(pdir/'gt_ppc.vtk')

        center = gt_full.mean(0).astype(np.float32)
        scale = compute_scale(gt_full)
        gt_local_full = np.clip(points_to_local(gt_full, center, scale), -1, 1)
        aux_occ = make_aux_occupancy(gt_local_full)
        gap_occ = make_gap_occupancy(gt_local_full)

        n = len(gt_full)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        gt_world = gt_full[sel].astype(np.float32)
        gt_local = gt_local_full[sel].astype(np.float32)

        if self.augment:
            for drr in [drr_ap, drr_lp]:
                drr[:] = np.clip(drr * (1 + (np.random.rand()*2-1)*0.15), 0, 1)
            if np.random.rand() < 0.5:
                drr_ap = drr_ap[:,::-1].copy()
                drr_lp = drr_lp[:,::-1].copy()
                P_ap = flip_projection_matrix_horizontal(P_ap)
                P_lp = flip_projection_matrix_horizontal(P_lp)

        return {
            'drr_ap': self.img_norm(torch.from_numpy(drr_ap).unsqueeze(0).float()),
            'drr_lp': self.img_norm(torch.from_numpy(drr_lp).unsqueeze(0).float()),
            'P_ap': torch.from_numpy(P_ap).float(),
            'P_lp': torch.from_numpy(P_lp).float(),
            'gt_ppc_world': torch.from_numpy(gt_world).float(),
            'gt_ppc_local': torch.from_numpy(gt_local).float(),
            'aux_occ': torch.from_numpy(aux_occ).float(),
            'gap_occ': torch.from_numpy(gap_occ).float(),
            'center': torch.from_numpy(center).float(),
            'scale': torch.from_numpy(scale).float(),
            'patient_id': pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f'Split: train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}')

Split: train=829 val=103 test=105


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# V8 GENERATOR (frozen) — copied from v8 for self-contained script
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=ENC_CHANNELS, pretrained=PRETRAINED):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        c1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad(): c1.weight[:] = base.conv1.weight.mean(1, keepdim=True)
        base.conv1 = c1
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1, self.layer2, self.layer3 = base.layer1, base.layer2, base.layer3
        self.proj = nn.Conv2d(256, out_ch, 1)
    def forward(self, x):
        return self.proj(self.layer3(self.layer2(self.layer1(self.stem(x)))))

class FeatureLift(nn.Module):
    def __init__(self, in_ch=ENC_CHANNELS, out_ch=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1, in_ch, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, f2d):
        B, C, H, W = f2d.shape
        vol = f2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        return self.refine(vol + self.depth_embed)

class BiplanarFusion(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, out_ch=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_ch*2, out_ch, 1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, ap, lp):
        return self.fuse(torch.cat([ap.permute(0,1,4,2,3).contiguous(),
                                    lp.permute(0,1,2,4,3).contiguous()], 1))

class Block3D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ic,oc,3,padding=1), nn.GroupNorm(8,oc), nn.GELU(),
            nn.Conv3d(oc,oc,3,padding=1), nn.GroupNorm(8,oc), nn.GELU())
    def forward(self, x): return self.block(x)

class CoarseUNet3D(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, fc=DEC_CHANNELS):
        super().__init__()
        self.enc1 = Block3D(in_ch, 96); self.down1 = nn.Conv3d(96, 160, 2, stride=2)
        self.enc2 = Block3D(160, 160);  self.down2 = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2 = nn.ConvTranspose3d(224, 160, 2, stride=2); self.dec2 = Block3D(320, 160)
        self.up1 = nn.ConvTranspose3d(160, 96, 2, stride=2);  self.dec1 = Block3D(192, fc)
        self.aux_head = nn.Sequential(nn.Conv3d(fc, fc//2, 3, padding=1), nn.GELU(), nn.Conv3d(fc//2, 1, 1))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.down1(e1))
        b = self.bottleneck(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        aux = F.interpolate(self.aux_head(d1), size=(AUX_VOL_SIZE,)*3, mode='trilinear', align_corners=True)
        return d1, aux

def project_points(P, pts, img_size=IMG_SIZE):
    B, N, _ = pts.shape
    homog = torch.cat([pts, torch.ones(B,N,1,device=pts.device,dtype=pts.dtype)], -1)
    uvw = torch.bmm(homog, P.transpose(1,2))
    z = uvw[...,2:3].clamp(min=1e-6)
    uv = uvw[...,:2] / z
    return uv, (uv/(img_size-1))*2-1, torch.log(z)

def sample_2d_features(fm, uvn):
    return F.grid_sample(fm, uvn.view(fm.shape[0],-1,1,2), mode='bilinear',
                         align_corners=True, padding_mode='border').squeeze(-1).transpose(1,2)

def sample_3d_features(vf, pl):
    grid = torch.stack([pl[...,2], pl[...,1], pl[...,0]], -1).view(pl.shape[0],-1,1,1,3)
    return F.grid_sample(vf, grid, mode='bilinear', align_corners=True,
                         padding_mode='border').squeeze(-1).squeeze(-1).transpose(1,2)

class QueryInitializerV8(nn.Module):
    def __init__(self, n_points=N_POINTS, fc=DEC_CHANNELS):
        super().__init__()
        grid = np.stack(np.meshgrid(np.linspace(-.8,.8,16), np.linspace(-.8,.8,16),
                                     np.linspace(-.9,.9,20), indexing='ij'), -1).reshape(-1,3).astype(np.float32)
        self.register_buffer('base_queries', torch.from_numpy(grid))
        self.n_points = n_points
        self.global_head = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                          nn.Linear(fc, 256), nn.GELU(), nn.Linear(256, n_points*3))
    def forward(self, vf, aux_occ=None):
        B = vf.shape[0]
        off = self.global_head(vf).view(B, self.n_points, 3)
        raw = self.base_queries[None] + 0.20 * torch.tanh(off)
        if aux_occ is not None:
            gs = torch.stack([raw[...,2], raw[...,1], raw[...,0]], -1).view(B, self.n_points, 1, 1, 3)
            gate = torch.sigmoid(F.grid_sample(aux_occ, gs, mode='bilinear',
                                                align_corners=True, padding_mode='border').view(B, self.n_points)).unsqueeze(-1)
            raw = self.base_queries[None] + gate * 0.25 * torch.tanh(off)
        return torch.clamp(raw, -1, 1)

class RefinementStageV8(nn.Module):
    def __init__(self, f2d=ENC_CHANNELS, f3d=DEC_CHANNELS, h=QUERY_DIM):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(f2d*2+f3d+3+2+2+1, h), nn.GELU(),
                                  nn.Linear(h, h), nn.GELU(), nn.Linear(h, 3))
    def forward(self, q, vf, fap, flp, Pap, Plp, c, s, aux_occ=None):
        qw = local_to_world(q, c, s)
        _, ua, _ = project_points(Pap, qw); _, ul, _ = project_points(Plp, qw)
        B, N, _ = q.shape
        if aux_occ is not None:
            gs = torch.stack([q[...,2], q[...,1], q[...,0]], -1).view(B,N,1,1,3)
            ov = F.grid_sample(aux_occ, gs, mode='bilinear', align_corners=True, padding_mode='border').view(B,N,1)
        else: ov = torch.zeros(B,N,1,device=q.device)
        x = torch.cat([sample_3d_features(vf,q), sample_2d_features(fap,ua),
                        sample_2d_features(flp,ul), q, ua, ul, ov], -1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q + delta, {'delta': delta}

class PPCNetV8(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D(); self.encoder_lp = Encoder2D()
        self.lift_ap = FeatureLift(); self.lift_lp = FeatureLift()
        self.fusion = BiplanarFusion(); self.coarse3d = CoarseUNet3D()
        self.query_init = QueryInitializerV8()
        self.stages = nn.ModuleList([RefinementStageV8() for _ in range(N_REFINE_STAGES)])
    def forward(self, dap, dlp, Pap, Plp, c, s):
        fap = self.encoder_ap(dap); flp = self.encoder_lp(dlp)
        vap = self.lift_ap(fap); vlp = self.lift_lp(flp)
        fused = self.fusion(vap, vlp)
        vf, aux = self.coarse3d(fused)
        q = self.query_init(vf, aux)
        sa = []
        for stage in self.stages:
            q, a = stage(q, vf, fap, flp, Pap, Plp, c, s, aux)
            sa.append(a)
        q = torch.clamp(q, -1, 1)
        return {'pred_local': q, 'pred_world': local_to_world(q, c, s),
                'aux_occ_logits': aux, 'stage_aux': sa}

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# SN-GAN DISCRIMINATOR — spectral norm instead of gradient penalty (saves ~25GB)
# Uses subsampled points (1024) and smaller k=10 for memory efficiency
# ══════════════════════════════════════════════════════════════════════════════

def knn_graph(x, k=10):
    """Build k-NN graph. x: (B, N, 3) → idx: (B, N, k)"""
    d = torch.cdist(x, x)  # (B, N, N)
    _, idx = d.topk(k+1, dim=-1, largest=False)
    return idx[:, :, 1:]

def get_edge_features(x, idx):
    B, N, C = x.shape
    k = idx.shape[-1]
    nb = torch.gather(x.unsqueeze(2).expand(B,N,N,C), 2,
                      idx.unsqueeze(-1).expand(B,N,k,C))
    return torch.cat([x.unsqueeze(2).expand_as(nb), nb - x.unsqueeze(2).expand_as(nb)], -1)

class EdgeConv(nn.Module):
    """Plain EdgeConv for generator/refiner (no spectral norm)."""
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Linear(in_c * 2, out_c), nn.LeakyReLU(0.2),
            nn.Linear(out_c, out_c), nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        edge_feat = get_edge_features(x, idx)
        return self.mlp(edge_feat).max(dim=2)[0]

class EdgeConvSN(nn.Module):
    """EdgeConv with spectral normalization for discriminator."""
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(in_c * 2, out_c)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(out_c, out_c)), nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        edge_feat = get_edge_features(x, idx)
        return self.mlp(edge_feat).max(dim=2)[0]

class PointDiscriminator(nn.Module):
    """SN-GAN discriminator (spectral norm = Lipschitz constraint without GP).
    Input: (B, D_SUBSAMPLE, 3) subsampled points in LOCAL [-1,1] space.
    Output: (B, 1) realness score.
    """
    def __init__(self, k=10):
        super().__init__()
        self.ec1 = EdgeConvSN(3, 64, k)
        self.ec2 = EdgeConvSN(64, 128, k)
        self.global_mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(64 + 128, 256)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(256, 128)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(128, 1)))

    def forward(self, x):
        f1 = self.ec1(x)                     # (B, N, 64)
        f2 = self.ec2(f1)                    # (B, N, 128)
        g = torch.cat([f1, f2], -1)          # (B, N, 192)
        g = g.max(dim=1)[0]                  # (B, 192) global
        return self.global_mlp(g)            # (B, 1)

def subsample_points(pts, n=D_SUBSAMPLE):
    """Randomly subsample points for discriminator input."""
    B, N, C = pts.shape
    idx = torch.randint(0, N, (B, n), device=pts.device)
    return torch.gather(pts, 1, idx.unsqueeze(-1).expand(B, n, C))

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# POINT REFINER — lightweight residual network on top of v8 output
# ══════════════════════════════════════════════════════════════════════════════

class PointRefiner(nn.Module):
    """Takes coarse point cloud (B, N, 3) and produces per-point offsets.
    Uses local neighborhood features (EdgeConv) to understand structure.
    """
    def __init__(self, k=16):
        super().__init__()
        self.ec1 = EdgeConv(3, 64, k)
        self.ec2 = EdgeConv(64, 128, k)
        self.offset_head = nn.Sequential(
            nn.Linear(64 + 128 + 3, 128), nn.GELU(),
            nn.Linear(128, 64), nn.GELU(),
            nn.Linear(64, 3))
        # Confidence head: predict per-point "keep" probability
        self.conf_head = nn.Sequential(
            nn.Linear(64 + 128 + 3, 64), nn.GELU(),
            nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, pts_local):
        """Returns refined points and per-point confidence."""
        f1 = self.ec1(pts_local)             # (B, N, 64)
        f2 = self.ec2(f1)                    # (B, N, 128)
        feat = torch.cat([f1, f2, pts_local], -1)  # (B, N, 195)
        offset = 0.08 * torch.tanh(self.offset_head(feat))  # small offsets
        conf = self.conf_head(feat)          # (B, N, 1) keep probability
        refined = torch.clamp(pts_local + offset, -1, 1)
        return refined, conf.squeeze(-1), offset

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def pairwise_sqdist(x, y):
    x2 = (x**2).sum(-1, keepdim=True)
    y2 = (y**2).sum(-1).unsqueeze(1)
    return (x2 + y2 - 2 * torch.bmm(x, y.transpose(1,2))).clamp_min(0)

def chamfer_loss(pred, gt):
    d2 = pairwise_sqdist(pred, gt)
    return 0.5 * (d2.min(2)[0].mean() + d2.min(1)[0].mean())

def gap_penalty_loss(pred_local, gap_occ_gt):
    B, N, _ = pred_local.shape
    occ = gap_occ_gt.unsqueeze(1)
    grid = torch.stack([pred_local[...,2], pred_local[...,1], pred_local[...,0]], -1)
    grid = grid.view(B, N, 1, 1, 3)
    sampled = F.grid_sample(occ, grid, mode='bilinear', align_corners=True, padding_mode='border')
    return (1.0 - sampled.view(B, N)).clamp(min=0).mean()

def soft_axial_density_loss(pred_l, gt_l, n_bins=60, sigma=0.025):
    centers = torch.linspace(-1, 1, n_bins, device=pred_l.device)
    p_kde = torch.exp(-0.5*((pred_l[...,2].unsqueeze(-1) - centers)/sigma)**2).sum(1)
    g_kde = torch.exp(-0.5*((gt_l[...,2].unsqueeze(-1) - centers)/sigma)**2).sum(1)
    p_d = p_kde / (p_kde.sum(1, keepdim=True) + 1e-8)
    g_d = g_kde / (g_kde.sum(1, keepdim=True) + 1e-8)
    return (p_d - g_d).abs().sum(1).mean()

def sliced_wasserstein_loss(pred, gt, n_proj=50):
    dirs = F.normalize(torch.randn(n_proj, 3, device=pred.device), -1)
    pp = torch.einsum('bnd,pd->bnp', pred, dirs)
    gp = torch.einsum('bnd,pd->bnp', gt, dirs)
    return ((pp.sort(1)[0] - gp.sort(1)[0])**2).mean()

# gradient_penalty REMOVED — replaced by spectral normalization
# SN provides the same Lipschitz constraint with ZERO extra memory

def confidence_loss(conf, pred_local, gap_occ_gt):
    """Teach confidence head: low confidence for points in gaps."""
    B, N, _ = pred_local.shape
    occ = gap_occ_gt.unsqueeze(1)
    grid = torch.stack([pred_local[...,2], pred_local[...,1], pred_local[...,0]], -1)
    grid = grid.view(B, N, 1, 1, 3)
    occ_at = F.grid_sample(occ, grid, mode='bilinear', align_corners=True,
                            padding_mode='border').view(B, N)
    # conf should be high where occ=1 (bone) and low where occ=0 (gap)
    return F.binary_cross_entropy(conf, occ_at.detach())

def chamfer_metrics_np(pred, gt):
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:,None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th):
        p, r = (dp<=th).mean(), (dg<=th).mean()
        return 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return {'chamfer_mm': float(.5*(dp.mean()+dg.mean())),
            'fscore_1mm': float(fs(1)), 'fscore_2mm': float(fs(2)),
            'fscore_5mm': float(fs(5)),
            'hausdorff_95': float(max(np.percentile(dp,95), np.percentile(dg,95)))}

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def collate_fn(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(batch):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}

# Load pre-trained v8 generator (FROZEN)
print('Loading pre-trained v8 generator...')
generator = PPCNetV8().to(device)
v8_ckpt = V8_CKPT_DIR / 'best_chamfer_checkpoint.pth'
if not v8_ckpt.exists():
    v8_ckpt = V8_CKPT_DIR / 'best_checkpoint.pth'
if not v8_ckpt.exists():
    v8_ckpt = V8_CKPT_DIR / 'latest_checkpoint.pth'
state = torch.load(v8_ckpt, map_location=device, weights_only=False)
generator.load_state_dict(state['model'])
generator.eval()
for p in generator.parameters(): p.requires_grad_(False)
print(f'  Loaded v8 epoch {state["epoch"]+1}')

# Create refiner and discriminator (k=10 for memory efficiency)
refiner = PointRefiner(k=16).to(device)
discriminator = PointDiscriminator(k=10).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Refiner params:       {count_params(refiner)/1e6:.2f} M')
print(f'Discriminator params: {count_params(discriminator)/1e6:.2f} M')

opt_G = torch.optim.AdamW(refiner.parameters(), lr=LR_G, betas=(0.5, 0.999), weight_decay=1e-5)
opt_D = torch.optim.AdamW(discriminator.parameters(), lr=LR_D, betas=(0.5, 0.999), weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')

train_ds = LumbarDatasetV9(train_ids, augment=True)
val_ds   = LumbarDatasetV9(val_ids, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

print(f'Train: {len(train_ds)} → {len(train_loader)} batches')
print(f'Val  : {len(val_ds)} → {len(val_loader)} batches')

def save_ckpt(path, epoch, best_val, history):
    torch.save({
        'refiner': refiner.state_dict(), 'discriminator': discriminator.state_dict(),
        'opt_G': opt_G.state_dict(), 'opt_D': opt_D.state_dict(),
        'epoch': epoch, 'best_val_loss': best_val, 'history': history,
        'config': {'version': 'v9_gan_refiner'}}, path.with_suffix('.pth.tmp'))
    path.with_suffix('.pth.tmp').replace(path)

best_val = float('inf')
best_chamfer = float('inf')
history = []

print(f'\n{"="*60}')
print('Starting GAN training...')
print(f'{"="*60}\n')

for epoch in range(GAN_EPOCHS):
    refiner.train(); discriminator.train()
    acc = {'d_loss': 0, 'g_loss': 0, 'chamfer': 0, 'gap': 0, 'axial': 0, 'conf': 0}
    n_bat = 0

    for bi, batch in enumerate(train_loader, 1):
        batch = to_dev(batch)

        # ── Generate coarse point cloud with frozen v8 ────────────────────
        with torch.no_grad():
            v8_out = generator(batch['drr_ap'], batch['drr_lp'],
                               batch['P_ap'], batch['P_lp'],
                               batch['center'], batch['scale'])
            coarse_local = v8_out['pred_local'].detach()

        gt_local = batch['gt_ppc_local']

        # ══════════════════════════════════════════════════════════════════
        # TRAIN DISCRIMINATOR (N_CRITIC steps per G step)
        # SN-GAN: hinge loss, no gradient penalty, subsampled points
        # ══════════════════════════════════════════════════════════════════
        if bi % (N_CRITIC + 1) != 0:
            opt_D.zero_grad(set_to_none=True)
            with torch.no_grad():
                refined, _, _ = refiner(coarse_local)

            # Subsample to D_SUBSAMPLE points for memory efficiency
            gt_sub = subsample_points(gt_local, D_SUBSAMPLE)
            fake_sub = subsample_points(refined.detach(), D_SUBSAMPLE)

            d_real = discriminator(gt_sub)
            d_fake = discriminator(fake_sub)
            # Hinge loss (SN-GAN standard)
            d_loss = (F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake).mean())

            d_loss.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP)
            opt_D.step()
            acc['d_loss'] += float(d_loss.detach().cpu())

        # ══════════════════════════════════════════════════════════════════
        # TRAIN REFINER (Generator)
        # ══════════════════════════════════════════════════════════════════
        else:
            opt_G.zero_grad(set_to_none=True)
            refined, conf, offset = refiner(coarse_local)

            # Adversarial loss (fool discriminator) — subsample for memory
            ref_sub = subsample_points(refined, D_SUBSAMPLE)
            g_adv = -discriminator(ref_sub).mean()

            # Reconstruction losses (in LOCAL space)
            g_ch  = chamfer_loss(refined, gt_local)
            g_gap = gap_penalty_loss(refined, batch['gap_occ'])
            g_ax  = soft_axial_density_loss(refined, gt_local)
            g_sw  = sliced_wasserstein_loss(refined, gt_local)
            g_conf = confidence_loss(conf, coarse_local, batch['gap_occ'])

            # Offset regularization
            g_off = offset.abs().mean()

            g_loss = (W_ADV * g_adv + W_CHAMFER_R * g_ch + W_GAP_R * g_gap
                     + W_AXIAL_R * g_ax + W_SW_R * g_sw + 0.2 * g_conf + 0.001 * g_off)

            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(refiner.parameters(), GRAD_CLIP)
            opt_G.step()

            acc['g_loss'] += float(g_loss.detach().cpu())
            acc['chamfer'] += float(g_ch.detach().cpu())
            acc['gap'] += float(g_gap.detach().cpu())
            acc['axial'] += float(g_ax.detach().cpu())
            acc['conf'] += float(g_conf.detach().cpu())

        n_bat += 1
        if bi % 100 == 0 or bi == len(train_loader):
            print(f"  [Ep {epoch+1:3d}/{GAN_EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"D={acc['d_loss']/max(1,n_bat):.4f} G={acc['g_loss']/max(1,n_bat):.4f} "
                  f"gap={acc['gap']/max(1,n_bat):.4f}")

    # ── Validation ────────────────────────────────────────────────────────
    refiner.eval()
    metrics = []; n_done = 0
    with torch.no_grad():
        for batch in val_loader:
            if n_done >= MAX_EVAL: break
            batch = to_dev(batch)
            v8_out = generator(batch['drr_ap'], batch['drr_lp'],
                               batch['P_ap'], batch['P_lp'],
                               batch['center'], batch['scale'])
            coarse_local = v8_out['pred_local']
            refined, conf, _ = refiner(coarse_local)

            # Apply confidence filtering: keep high-confidence points,
            # replace low-confidence with nearest high-confidence point
            ref_world = local_to_world(refined, batch['center'], batch['scale'])
            gt_w = batch['gt_ppc_world'].cpu().numpy()
            pred = ref_world.cpu().numpy()

            for b in range(pred.shape[0]):
                if n_done >= MAX_EVAL: break
                c = conf[b].cpu().numpy()
                pts = pred[b]
                # Filter: keep points with conf > 0.3, duplicate high-conf to fill
                mask = c > 0.3
                if mask.sum() > 100:
                    good_pts = pts[mask]
                    n_need = N_POINTS - len(good_pts)
                    if n_need > 0:
                        extra = good_pts[np.random.choice(len(good_pts), n_need, replace=True)]
                        pts = np.concatenate([good_pts, extra], 0)
                    else:
                        pts = good_pts[:N_POINTS]
                metrics.append(chamfer_metrics_np(pts, gt_w[b]))
                n_done += 1

    if metrics:
        mean_ch = np.mean([m['chamfer_mm'] for m in metrics])
        f1 = np.mean([m['fscore_1mm'] for m in metrics])
        f2 = np.mean([m['fscore_2mm'] for m in metrics])
        f5 = np.mean([m['fscore_5mm'] for m in metrics])
        hd = np.mean([m['hausdorff_95'] for m in metrics])

        print(f"[Epoch {epoch+1}] Chamfer={mean_ch:.3f}mm F@1={f1:.3f} F@2={f2:.3f} "
              f"F@5={f5:.3f} HD95={hd:.3f}")

        rec = {'epoch': epoch+1, 'chamfer_mm': mean_ch, 'f1': f1, 'f2': f2, 'f5': f5, 'hd95': hd}
        history.append(rec)
        append_training_log(TRAINING_LOG, rec)

        save_ckpt(CKPT_PATH, epoch, best_val, history)
        if mean_ch < best_chamfer:
            best_chamfer = mean_ch
            save_ckpt(CKPT_DIR / 'best_chamfer.pth', epoch, best_val, history)
            print(f"  ✓ New best Chamfer: {best_chamfer:.3f} mm")

print(f'\nGAN training complete. Best Chamfer: {best_chamfer:.3f} mm')

Loading pre-trained v8 generator...
  Loaded v8 epoch 79
Refiner params:       0.08 M
Discriminator params: 0.12 M
Train: 829 → 415 batches
Val  : 103 → 52 batches

Starting GAN training...

  [Ep   1/150] 100/415 D=1.4992 G=5.1403 gap=0.2181
  [Ep   1/150] 200/415 D=1.4954 G=2.7657 gap=0.2182
  [Ep   1/150] 300/415 D=1.4859 G=2.1311 gap=0.2184
  [Ep   1/150] 400/415 D=1.4747 G=6.3682 gap=0.2183
  [Ep   1/150] 415/415 D=1.4767 G=6.1891 gap=0.2166
[Epoch 1] Chamfer=2.214mm F@1=0.120 F@2=0.532 F@5=0.954 HD95=5.439
  ✓ New best Chamfer: 2.214 mm
  [Ep   2/150] 100/415 D=1.4236 G=6.6518 gap=0.2201
  [Ep   2/150] 200/415 D=1.4202 G=6.2977 gap=0.2189
  [Ep   2/150] 300/415 D=1.4165 G=4.5437 gap=0.2187
  [Ep   2/150] 400/415 D=1.4132 G=3.6626 gap=0.2184
  [Ep   2/150] 415/415 D=1.4162 G=3.5395 gap=0.2169
[Epoch 2] Chamfer=2.211mm F@1=0.121 F@2=0.533 F@5=0.954 HD95=5.425
  ✓ New best Chamfer: 2.211 mm
  [Ep   3/150] 100/415 D=1.3943 G=1.3537 gap=0.2189
  [Ep   3/150] 200/415 D=1.3930 G=6.2366 

  [Ep  24/150] 415/415 D=1.3853 G=996.9862 gap=0.2194
[Epoch 24] Chamfer=2.273mm F@1=0.110 F@2=0.505 F@5=0.955 HD95=5.178
  [Ep  25/150] 100/415 D=1.3872 G=24.6456 gap=0.2224
  [Ep  25/150] 200/415 D=1.3847 G=12.6146 gap=0.2218
  [Ep  25/150] 300/415 D=1.3848 G=1022.2921 gap=0.2216
  [Ep  25/150] 400/415 D=1.3868 G=766.8676 gap=0.2217
  [Ep  25/150] 415/415 D=1.3903 G=741.0773 gap=0.2201
[Epoch 25] Chamfer=2.291mm F@1=0.108 F@2=0.499 F@5=0.954 HD95=5.191
  [Ep  26/150] 100/415 D=1.3890 G=60.4971 gap=0.2210
  [Ep  26/150] 200/415 D=1.3862 G=58.2194 gap=0.2210
  [Ep  26/150] 300/415 D=1.3884 G=39.7961 gap=0.2207
  [Ep  26/150] 400/415 D=1.3905 G=29.9589 gap=0.2211
  [Ep  26/150] 415/415 D=1.3937 G=38.6826 gap=0.2195
[Epoch 26] Chamfer=2.293mm F@1=0.108 F@2=0.499 F@5=0.954 HD95=5.192
  [Ep  27/150] 100/415 D=1.3921 G=0.7431 gap=0.2215
  [Ep  27/150] 200/415 D=1.3884 G=25.0535 gap=0.2212
  [Ep  27/150] 300/415 D=1.3881 G=19.1020 gap=0.2218
  [Ep  27/150] 400/415 D=1.3875 G=14.7801 gap=0.22

  [Ep  50/150] 400/415 D=1.3886 G=6.4623 gap=0.2218
  [Ep  50/150] 415/415 D=1.3920 G=6.2425 gap=0.2202
[Epoch 50] Chamfer=2.324mm F@1=0.105 F@2=0.489 F@5=0.952 HD95=5.222
  [Ep  51/150] 100/415 D=1.3879 G=0.4377 gap=0.2222
  [Ep  51/150] 200/415 D=1.3909 G=34.4970 gap=0.2224
  [Ep  51/150] 300/415 D=1.3912 G=23.1097 gap=0.2223
  [Ep  51/150] 400/415 D=1.3911 G=259.6101 gap=0.2221
  [Ep  51/150] 415/415 D=1.3949 G=250.2334 gap=0.2205
[Epoch 51] Chamfer=2.325mm F@1=0.105 F@2=0.489 F@5=0.952 HD95=5.236
  [Ep  52/150] 100/415 D=1.3900 G=0.6572 gap=0.2212
  [Ep  52/150] 200/415 D=1.3909 G=1.1756 gap=0.2218
  [Ep  52/150] 300/415 D=1.3928 G=0.9746 gap=0.2221
  [Ep  52/150] 400/415 D=1.3927 G=0.9185 gap=0.2220
  [Ep  52/150] 415/415 D=1.3956 G=0.8905 gap=0.2204
[Epoch 52] Chamfer=2.324mm F@1=0.105 F@2=0.489 F@5=0.952 HD95=5.231
  [Ep  53/150] 100/415 D=1.3910 G=24.6017 gap=0.2227
  [Ep  53/150] 200/415 D=1.3882 G=12.4391 gap=0.2224
  [Ep  53/150] 300/415 D=1.3928 G=8.4996 gap=0.2222
  [Ep  5

  [Ep  76/150] 300/415 D=1.3959 G=3.5761 gap=0.2223
  [Ep  76/150] 400/415 D=1.3941 G=2.9575 gap=0.2221
  [Ep  76/150] 415/415 D=1.3978 G=2.8558 gap=0.2204
[Epoch 76] Chamfer=2.337mm F@1=0.104 F@2=0.485 F@5=0.951 HD95=5.244
  [Ep  77/150] 100/415 D=1.3949 G=231.0826 gap=0.2227
  [Ep  77/150] 200/415 D=1.3952 G=128.6480 gap=0.2225
  [Ep  77/150] 300/415 D=1.3933 G=86.2464 gap=0.2226
  [Ep  77/150] 400/415 D=1.3930 G=65.0329 gap=0.2225
  [Ep  77/150] 415/415 D=1.3960 G=62.6902 gap=0.2209
[Epoch 77] Chamfer=2.334mm F@1=0.104 F@2=0.486 F@5=0.951 HD95=5.237
  [Ep  78/150] 100/415 D=1.3914 G=0.3392 gap=0.2217
  [Ep  78/150] 200/415 D=1.3948 G=0.4949 gap=0.2226
  [Ep  78/150] 300/415 D=1.3948 G=1.4759 gap=0.2226
  [Ep  78/150] 400/415 D=1.3949 G=2.5192 gap=0.2227
  [Ep  78/150] 415/415 D=1.3979 G=2.4437 gap=0.2211
[Epoch 78] Chamfer=2.331mm F@1=0.104 F@2=0.487 F@5=0.951 HD95=5.238
  [Ep  79/150] 100/415 D=1.3966 G=1940.8978 gap=0.2223
  [Ep  79/150] 200/415 D=1.4002 G=970.6269 gap=0.2226
  [E

[Epoch 101] Chamfer=2.347mm F@1=0.103 F@2=0.482 F@5=0.950 HD95=5.266
  [Ep 102/150] 100/415 D=1.3983 G=0.4955 gap=0.2224
  [Ep 102/150] 200/415 D=1.3981 G=0.8309 gap=0.2231
  [Ep 102/150] 300/415 D=1.3988 G=0.6679 gap=0.2233
  [Ep 102/150] 400/415 D=1.3987 G=0.6657 gap=0.2233
  [Ep 102/150] 415/415 D=1.4019 G=0.6483 gap=0.2217
[Epoch 102] Chamfer=2.346mm F@1=0.103 F@2=0.483 F@5=0.950 HD95=5.269
  [Ep 103/150] 100/415 D=1.3926 G=14.6014 gap=0.2213
  [Ep 103/150] 200/415 D=1.3976 G=11.2777 gap=0.2213
  [Ep 103/150] 300/415 D=1.3989 G=9.0614 gap=0.2222
  [Ep 103/150] 400/415 D=1.3984 G=12.6573 gap=0.2226
  [Ep 103/150] 415/415 D=1.4025 G=12.2048 gap=0.2210
[Epoch 103] Chamfer=2.352mm F@1=0.103 F@2=0.481 F@5=0.950 HD95=5.274
  [Ep 104/150] 100/415 D=1.3977 G=3.7199 gap=0.2232
  [Ep 104/150] 200/415 D=1.3978 G=10.4919 gap=0.2230
  [Ep 104/150] 300/415 D=1.4004 G=8.8094 gap=0.2230
  [Ep 104/150] 400/415 D=1.3991 G=7.0125 gap=0.2230
  [Ep 104/150] 415/415 D=1.4031 G=6.7640 gap=0.2213
[Epoch 1

  [Ep 127/150] 400/415 D=1.4016 G=17.7713 gap=0.2239
  [Ep 127/150] 415/415 D=1.4050 G=17.1417 gap=0.2223
[Epoch 127] Chamfer=2.377mm F@1=0.100 F@2=0.474 F@5=0.948 HD95=5.312
  [Ep 128/150] 100/415 D=1.4009 G=0.4044 gap=0.2247
  [Ep 128/150] 200/415 D=1.4011 G=0.4831 gap=0.2244
  [Ep 128/150] 300/415 D=1.4008 G=0.8590 gap=0.2247
  [Ep 128/150] 400/415 D=1.3999 G=1.2235 gap=0.2245
  [Ep 128/150] 415/415 D=1.4029 G=1.2370 gap=0.2229
[Epoch 129] Chamfer=2.381mm F@1=0.100 F@2=0.474 F@5=0.947 HD95=5.327
  [Ep 130/150] 100/415 D=1.4018 G=1.3264 gap=0.2242
  [Ep 130/150] 200/415 D=1.4007 G=0.7799 gap=0.2242
  [Ep 130/150] 300/415 D=1.4012 G=1722.7990 gap=0.2231
  [Ep 130/150] 400/415 D=1.4012 G=1292.6021 gap=0.2235
  [Ep 130/150] 415/415 D=1.4048 G=1245.8916 gap=0.2220
[Epoch 130] Chamfer=2.370mm F@1=0.101 F@2=0.477 F@5=0.948 HD95=5.317
  [Ep 131/150] 100/415 D=1.3894 G=0.6036 gap=0.2233
  [Ep 131/150] 200/415 D=1.3957 G=0.4203 gap=0.2237
  [Ep 131/150] 300/415 D=1.3985 G=6.7604 gap=0.2242
  

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print('\nLoading best GAN refiner for test evaluation...')
best_path = CKPT_DIR / 'best_chamfer.pth'
if best_path.exists():
    st = torch.load(best_path, map_location=device, weights_only=False)
    refiner.load_state_dict(st['refiner'])
refiner.eval()

test_ds = LumbarDatasetV9(test_ids, augment=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

all_metrics = []
with torch.no_grad():
    for batch in test_loader:
        batch = to_dev(batch)
        v8_out = generator(batch['drr_ap'], batch['drr_lp'],
                           batch['P_ap'], batch['P_lp'],
                           batch['center'], batch['scale'])
        refined, conf, _ = refiner(v8_out['pred_local'])
        ref_world = local_to_world(refined, batch['center'], batch['scale'])

        pred = ref_world.cpu().numpy()
        gt = batch['gt_ppc_world'].cpu().numpy()
        pids = batch['patient_id']

        for b in range(pred.shape[0]):
            c = conf[b].cpu().numpy()
            pts = pred[b]
            mask = c > 0.3
            if mask.sum() > 100:
                good = pts[mask]
                n_need = N_POINTS - len(good)
                if n_need > 0:
                    pts = np.concatenate([good, good[np.random.choice(len(good), n_need, True)]], 0)
                else:
                    pts = good[:N_POINTS]

            m = chamfer_metrics_np(pts, gt[b])
            m['patient_id'] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pts, RESULTS_DIR / f'{pids[b]}_pred_gan.vtk')

print(f'\n{"="*60}')
print(f'TEST SET RESULTS ({len(all_metrics)} patients)')
print(f'{"="*60}')
for k, lbl in [('chamfer_mm','Chamfer (mm)'), ('fscore_1mm','F@1mm'),
                ('fscore_2mm','F@2mm'), ('fscore_5mm','F@5mm'),
                ('hausdorff_95','HD95 (mm)')]:
    vals = [m[k] for m in all_metrics]
    print(f'  {lbl:<16} mean={np.mean(vals):.3f} std={np.std(vals):.3f} '
          f'median={np.median(vals):.3f}')

import csv
csv_path = RESULTS_DIR / 'test_results_gan.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['patient_id','chamfer_mm','fscore_1mm','fscore_2mm','fscore_5mm','hausdorff_95'])
    w.writeheader(); w.writerows(all_metrics)
print(f'Saved to {csv_path}')


Loading best GAN refiner for test evaluation...

TEST SET RESULTS (105 patients)
  Chamfer (mm)     mean=2.484 std=1.483 median=2.220
  F@1mm            mean=0.112 std=0.049 median=0.106
  F@2mm            mean=0.504 std=0.132 median=0.506
  F@5mm            mean=0.937 std=0.093 median=0.962
  HD95 (mm)        mean=6.726 std=7.264 median=5.312
Saved to /apps/app/pandu/ppc_network_v9_gan/results/test_results_gan.csv


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def predict_ppc_gan(drr_ap_path, drr_lp_path, p_ap_path, p_lp_path,
                    output_path, v8_ckpt_path=None, gan_ckpt_path=None,
                    center_mm=None, scale_mm=None, conf_threshold=0.3):
    """Full inference: v8 generator → GAN refiner → filtered output."""
    # Load v8
    if v8_ckpt_path is None:
        v8_ckpt_path = V8_CKPT_DIR / 'best_chamfer_checkpoint.pth'
    gen = PPCNetV8().to(device)
    gen.load_state_dict(torch.load(v8_ckpt_path, map_location=device, weights_only=False)['model'])
    gen.eval()

    # Load refiner
    if gan_ckpt_path is None:
        gan_ckpt_path = CKPT_DIR / 'best_chamfer.pth'
    ref = PointRefiner(k=16).to(device)
    ref.load_state_dict(torch.load(gan_ckpt_path, map_location=device, weights_only=False)['refiner'])
    ref.eval()

    img_norm = transforms.Normalize(mean=[0.485], std=[0.229])
    def _load(p):
        img = load_drr_image(p)
        return img_norm(torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()).to(device)

    dap = _load(drr_ap_path); dlp = _load(drr_lp_path)
    Pap = torch.from_numpy(load_projection_matrix(p_ap_path)).unsqueeze(0).float().to(device)
    Plp = torch.from_numpy(load_projection_matrix(p_lp_path)).unsqueeze(0).float().to(device)

    if center_mm is None: center_mm = [0,0,0]
    if scale_mm is None: scale_mm = [80,80,130]
    ct = torch.tensor([center_mm], dtype=torch.float32, device=device)
    st = torch.tensor([scale_mm], dtype=torch.float32, device=device)

    with torch.no_grad():
        v8_out = gen(dap, dlp, Pap, Plp, ct, st)
        refined, conf, _ = ref(v8_out['pred_local'])
        ref_world = local_to_world(refined, ct, st).squeeze(0).cpu().numpy()
        c = conf.squeeze(0).cpu().numpy()

    # Filter low-confidence (gap) points
    mask = c > conf_threshold
    if mask.sum() > 100:
        pts = ref_world[mask]
    else:
        pts = ref_world

    save_vtk_points(pts, output_path)
    print(f'Saved {len(pts)} points → {output_path}')
    return pts

print('\npredict_ppc_gan() ready.')


predict_ppc_gan() ready.
